# Altair声明式可视化

学习目标：用数据、图元和视觉编码描述图表；用聚合与堆叠表达总量和占比，用面积和热力图表达变化与分组差异；实现选择、高亮和联动过滤；保存可离线阅读的交互 HTML。

前置知识：Python 模块导入、字典与列表、pandas 表格选择和分组统计、均值与计数。

运行环境：Python 3.12、Altair 6.3；内嵌资源的 HTML 导出需要 vl-convert-python。

环境准备：见 [环境配置与运行说明](README.md)。Notebook 默认图形显示需要加载在线 JavaScript 资源；篇末另行保存内嵌资源版 HTML。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

本章是扩展内容，侧重声明式可视化：说明字段如何映射到图形，让 Altair 生成图表规格，再由浏览器绘制并处理交互。

阅读安排：图表单元按用途、数据、核心绘制、参数调整和成图展开。第 14～15 节组合与导出作为补充练习。

静态阅读：组合示例附近附有实际页面截图；运行本章或打开导出的 HTML 可操作交互。

## 1 从一张散点图开始

In [1]:
# 适用场景：散点图观察输入大小与请求耗时的关系，用颜色区分实现。
# 数据组织：每行一条教学模拟请求，两种实现 A、B 各 6 条，非真实性能实验。
# payload_kb 为输入大小（kB），latency_ms 为耗时（ms）；
# load 为 Low/Medium/High 有序负载，date 为模拟日期。
import altair as alt
import pandas as pd
from IPython.display import display

requests = pd.DataFrame(
    {
        "request_id": [f"r{i:02d}" for i in range(1, 13)],
        "method": ["A"] * 6 + ["B"] * 6,
        "payload_kb": [10, 20, 30, 40, 50, 60] * 2,
        "latency_ms": [8, 12, 17, 23, 29, 35, 7, 11, 14, 18, 22, 27],
        "load": ["Low", "Low", "Medium", "Medium", "High", "High"] * 2,
        "date": pd.to_datetime(
            [
                "2026-09-01",
                "2026-09-02",
                "2026-09-04",
                "2026-09-07",
                "2026-09-08",
                "2026-09-12",
            ]
            * 2
        ),
    }
)

# 最小绘制代码：Chart 接收表格，mark_point 画点，encode 规定字段与位置的映射。
# Q 表示定量值，N 表示无序类别；先用两个位置字段完成散点图。
points = alt.Chart(requests).mark_point().encode(x="payload_kb:Q", y="latency_ms:Q")

# 常用参数调整：color="method:N" 让两种实现使用不同颜色。
points = points.encode(color="method:N")

# 实际成图：每条请求对应一个点。display 显式展示图表，由浏览器绘制规格。
display(points)

alt.Chart(...)

## 2 把映射和外观分开

In [2]:
# 适用场景：继续调整分组散点图，使组别、单位和悬停读数更清楚。
# 数据组织：复用 requests；每个点仍是一条请求，没有做均值聚合。
# 颜色尺度 domain 给类别，range 按相同顺序给颜色，后续过滤也沿用这份映射。
method_color = alt.Color("method:N", title="Method").scale(
    domain=["A", "B"], range=["#0072B2", "#E69F00"]
)
# 最小绘制代码：沿用 Chart → mark_point → encode。
# 常用参数调整：filled/size 控制填充与大小；shape 增加点形区分，
# axis title 写单位，scale(domain=...) 固定范围；tooltip 列出悬停字段。
scatter = (
    alt.Chart(requests)
    .mark_point(filled=True, size=90)
    .encode(
        x=alt.X("payload_kb:Q").title("Payload (kB)").scale(domain=[0, 65]),
        y=alt.Y("latency_ms:Q").title("Latency (ms)").scale(domain=[0, 40]),
        color=method_color,
        shape=alt.Shape("method:N", title="Method").scale(domain=["A", "B"]),
        tooltip=["request_id:N", "method:N", "payload_kb:Q", "latency_ms:Q"],
    )
    .properties(width=380, height=240, title="Simulated requests")
)

# 实际成图：A 为蓝色、B 为橙色；悬停时用 request_id 对照原始记录。
display(scatter)

alt.Chart(...)

图表对象保存的是一份描述。to_dict 将其转为 Python 字典，能直接看到图元与字段类型；to_json 则生成 JSON 文本。这里只打印关键部分，避免整份数据淹没映射关系。

In [3]:
spec = scatter.to_dict()
print("图元：", spec["mark"])  # 预期：{'type': 'point', 'filled': True, 'size': 90}。
print("横轴编码：", spec["encoding"]["x"])  # 预期：field 为 payload_kb、type 为 quantitative、domain 为 [0, 65]。
print("颜色编码：", spec["encoding"]["color"])  # 预期：field 为 method、type 为 nominal，A/B 分别对应蓝/橙。
# 字典中的 field 是字段名，type 是编码类型；它们共同描述如何读这列数据。

图元： {'type': 'point', 'filled': True, 'size': 90}
横轴编码： {'field': 'payload_kb', 'scale': {'domain': [0, 65]}, 'title': 'Payload (kB)', 'type': 'quantitative'}
颜色编码： {'field': 'method', 'scale': {'domain': ['A', 'B'], 'range': ['#0072B2', '#E69F00']}, 'title': 'Method', 'type': 'nominal'}


## 3 根据含义声明字段类型

本章使用以下四种类型。它们描述测量含义，不等同于 Python 的存储类型；例如整数也可能只是类别编号。DataFrame 的自动推断不一定符合分析意图，本章显式标注类型。

| 类型 | 简写 | 中文名称／含义 | 本章示例 |
| --- | --- | --- | --- |
| quantitative | Q | 定量：大小与差值有数值含义 | 耗时 |
| nominal | N | 无序类别：区分组别 | 实现 A、B |
| ordinal | O | 有序类别：类别间有先后关系 | 负载等级 |
| temporal | T | 时间：日期或时刻 | 模拟日期 |

alt.hconcat 将图并排放置。下面保持同一批记录和相同纵轴，只改变输入大小的编码类型：Q 保留数值间距，N 把各取值当作离散标签。

In [4]:
numeric_x = scatter.properties(width=270, title="Q: numeric distance")
category_x = scatter.encode(
    x=alt.X("payload_kb:N").title("Payload labels (kB)").sort([10, 20, 30, 40, 50, 60])
).properties(width=270, title="N: separate categories")
display(alt.hconcat(numeric_x, category_x))
# 本表输入大小恰好等距；改变为不等距值时，Q 与 N 的位置差别会更明显。
# 将大小当作 N 会隐藏数值间隔；本例分析输入大小与耗时的关系时采用 Q。

alt.HConcatChart(...)

有序类别还需要说明顺序。仅写 O 不会自动理解 Low、Medium、High 的业务含义；sort 显式列出低、中、高的排列。每档有多条请求，点可能重叠，应结合悬停信息阅读。

In [5]:
load_chart = scatter.encode(
    x=alt.X("load:O").sort(["Low", "Medium", "High"]).title("Load level")
).properties(title="Ordered load levels")
display(load_chart)
# O 表达先后关系；相邻等级在图上的距离不表示负载差恰好相等。

alt.Chart(...)

### 3.1 用日期绘制折线图

In [6]:
# 适用场景：折线图查看各实现的耗时随日期怎样变化。
# 数据组织：复用 requests 和 method_color；date 是 pandas 日期列。
# 不带时区的日期按浏览器本地时间解释，真实跨时区时刻需先统一时区约定。
# 最小绘制代码：mark_line 配合 date:T，按 method 的颜色编码拆分曲线。
# 常用参数调整：point=True 显示观测点；axis format 控制日期标签，
# tickCount 请求刻度数量；tooltip 用完整日期帮助辨认记录。
timeline = (
    alt.Chart(requests)
    .mark_line(point=True)
    .encode(
        x=alt.X("date:T").title("Date (local)").axis(format="%m-%d", tickCount=6),
        y=alt.Y("latency_ms:Q").title("Latency (ms)"),
        color=method_color,
        tooltip=[alt.Tooltip("date:T", format="%Y-%m-%d"), "method:N", "latency_ms:Q"],
    )
    .properties(width=460, height=220, title="Uneven observation dates")
)

# 实际成图：09-08 到 09-12 的距离大于 09-01 到 09-02。
# 时间尺度保留不等间距，连线只连接已有观测。
display(timeline)

alt.Chart(...)

## 4 均值条形图与分组散点图

In [7]:
# 适用场景：水平条形图比较两种实现的平均耗时。
# 数据组织：复用 requests；mean(latency_ms) 求均值，count() 数请求记录。
# 编码中未聚合字段决定分组；本例 y、color 都是 method。
# 不随意把 request_id 加入非聚合编码，否则会改变一根柱对应的分组。
# 最小绘制代码：mark_bar 用 method 作类别位置、mean(latency_ms) 作柱长。
# 常用参数调整：复用 method_color；轴标题补 ms，tooltip 使用相同聚合口径。
mean_bars = (
    alt.Chart(requests)
    .mark_bar()
    .encode(
        x=alt.X("mean(latency_ms):Q").title("Mean latency (ms)"),
        y=alt.Y("method:N").title("Method"),
        color=method_color,
        tooltip=[
            "method:N",
            alt.Tooltip("mean(latency_ms):Q", format=".2f"),
            "count():Q",
        ],
    )
    .properties(width=380, height=140, title="All requests, grouped by method")
)
print(requests.groupby("method")["latency_ms"].agg(["size", "mean"]).round(2))

# 实际成图：每组各 6 条，柱长表示均值，不能替代组内原始散点。
display(mean_bars)

        size   mean
method             
A          6  20.67
B          6  16.50


alt.Chart(...)

### 4.1 用散点显示各负载档的均值

In [8]:
# 适用场景：分组均值散点图比较“实现 × 负载”各组合的平均耗时。
# 数据组织：复用 requests；transform_aggregate 显式按 method、load 分组，
# 生成 mean_ms 和 n 字段；每个组合包含两条原始请求。
# 最小绘制代码：先聚合，再 mark_point，将 load 和 mean_ms 放到两个坐标轴。
# 常用参数调整：sort 明确 Low、Medium、High 顺序；size 控制点大小，
# tooltip 同时显示均值与样本数，改变 groupby 就会改变统计问题。
load_means = (
    alt.Chart(requests)
    .transform_aggregate(
        mean_ms="mean(latency_ms)", n="count()", groupby=["method", "load"]
    )
    .mark_point(filled=True, size=110)
    .encode(
        x=alt.X("load:O").sort(["Low", "Medium", "High"]).title("Load level"),
        y=alt.Y("mean_ms:Q").title("Mean latency (ms)"),
        color=method_color,
        tooltip=["method:N", "load:O", "mean_ms:Q", "n:Q"],
    )
    .properties(width=380, height=220, title="Mean within each method and load")
)

# 实际成图：每点是两条请求的均值，点的高度不代表单条原始耗时。
display(load_means)

alt.Chart(...)

## 5 堆叠柱状图与百分比堆叠图

### 5.1 堆叠柱状图：总量与组成

In [9]:
# 适用场景：堆叠柱状图同时比较各记录日的总量与 A、B 组成。
# 数据组织：每行是某日某实现的模拟完成量，completed 单位为次，分量非负。
# 这是另一张汇总表，不是 requests 的统计结果；对 completed 求 sum，
# count() 只会数表格行。未记录日期未知，不能当作 0。
daily_counts = pd.DataFrame(
    {
        "date": pd.to_datetime(
            [
                "2026-09-01",
                "2026-09-01",
                "2026-09-02",
                "2026-09-02",
                "2026-09-04",
                "2026-09-04",
                "2026-09-07",
                "2026-09-07",
            ]
        ),
        "method": ["A", "B"] * 4,
        "completed": [12, 8, 18, 12, 16, 24, 30, 20],
    }
)
daily_counts["date_label"] = daily_counts["date"].dt.strftime("%m-%d")
print(daily_counts.to_string(index=False))
print(daily_counts.groupby("date")["completed"].sum())
# 四个记录日的总数依次为 20、30、40、50 次；每一天仍只有两行汇总记录。

# 最小绘制代码：mark_bar、日期位置、sum(completed) 高度和 method 颜色。
# 常用参数调整：stack="zero" 从 0 堆叠，order 固定 A、B 次序；
# date_label 是月日文本，O 与 sort 让记录日等距排列，不表示实际相隔天数。
# dt.strftime 只生成标签，原 date 列保留给时间轴和悬停。
stacked_bars = (
    alt.Chart(daily_counts)
    .mark_bar()
    .encode(
        x=alt.X("date_label:O")
        .title("Recorded date")
        .sort(["09-01", "09-02", "09-04", "09-07"])
        .axis(labelAngle=0),
        y=alt.Y("sum(completed):Q")
        .stack("zero")
        .title("Completed requests")
        .scale(domain=[0, 55]),
        color=method_color,
        order=alt.Order("method:N").sort("ascending"),
        tooltip=[
            alt.Tooltip("date:T", title="Date", format="%Y-%m-%d"),
            alt.Tooltip("method:N", title="Method"),
            alt.Tooltip("sum(completed):Q", title="Completed requests"),
        ],
    )
    .properties(width=380, height=220, title="Daily totals and components")
)

# 实际成图：09-01 的蓝段 12、橙段 8，总高 20；09-07 的总高 50。
# 分量读厚度，总量读上沿；悬停读取分量，不是该段上沿的累计高度。
display(stacked_bars)

      date method  completed date_label
2026-09-01      A         12      09-01
2026-09-01      B          8      09-01
2026-09-02      A         18      09-02
2026-09-02      B         12      09-02
2026-09-04      A         16      09-04
2026-09-04      B         24      09-04
2026-09-07      A         30      09-07
2026-09-07      B         20      09-07
date
2026-09-01    20
2026-09-02    30
2026-09-04    40
2026-09-07    50
Name: completed, dtype: int64


alt.Chart(...)

### 5.2 百分比堆叠图：比较组成比例

In [10]:
# 适用场景：百分比堆叠图比较不同日期的组成比例。
# 数据组织：复用 daily_counts 和 stacked_bars，输入仍是原始数量。
# 最小绘制代码：将堆叠规则改为 normalize，各分量除以所在日期的合计。
# 常用参数调整：比例轴设为 0～1，format=".0%" 显示整数百分数。
# 保留原图的颜色、顺序和数量提示；归一化发生在编码中，不改写数据。
normalized_bars = stacked_bars.encode(
    y=alt.Y("sum(completed):Q")
    .stack("normalize")
    .title("Share within recorded date")
    .scale(domain=[0, 1])
    .axis(format=".0%")
).properties(title="Composition; each recorded date = 100%")

# 实际成图：各柱总高为 100%，A 的占比依次为 60%、60%、40%、60%。
# 09-01 的 12/20 与 09-07 的 30/50 都是 60%，相同比例不代表完成量相同。
# 悬停保留原始数量，读取占比时分母是所在日期的合计。
display(normalized_bars)

alt.Chart(...)

## 6 面积图：从总量变化到分量变化

### 6.1 单个面积表达合计随时间变化

In [11]:
# 适用场景：面积图观察合计随时间的变化，曲线与基线之间填充区域。
# 数据组织：复用 daily_counts；不编码 method，就按日期对 A、B 求和。
# 最小绘制代码：mark_area 配合 date:T 和 sum(completed)。
# 常用参数调整：color 设灰色、opacity 控制透明度；point 显示日期标记，
# interpolate="linear" 用直线连接相邻记录，时间轴保留真实日期间距。
total_area = (
    alt.Chart(daily_counts)
    .mark_area(
        color="#6B7280", opacity=0.65, point={"color": "#6B7280"}, interpolate="linear"
    )
    .encode(
        x=alt.X("date:T").title("Date (local)").axis(format="%m-%d", tickCount=4),
        y=alt.Y("sum(completed):Q").title("Completed requests").scale(domain=[0, 55]),
        tooltip=[
            alt.Tooltip("date:T", title="Date", format="%Y-%m-%d"),
            alt.Tooltip("sum(completed):Q", title="Daily total"),
        ],
    )
    .properties(width=420, height=220, title="Total at each recorded date")
)

# 实际成图：四个标记高为 20、30、40、50，表示各日合计，不是跨日累计。
# 09-02 到 09-04 之间只是插值，不表示采集了 09-03；注意未观测的日期。
display(total_area)

alt.Chart(...)

### 6.2 颜色分组与 stack 共同生成堆叠面积

In [12]:
# 适用场景：堆叠面积图同时观察各日期总量与分量变化。
# 数据组织：复用 daily_counts、method_color，加入颜色后按“日期 × 实现”求和。
# 最小绘制代码：mark_area 加入 method 颜色与 stack="zero"。
# 常用参数调整：order 固定 A 在下、B 在上；point=True 标出各层上沿的记录日期，
# 无需手算上下界；tooltip 显示原始分量，linear 仅连接已有日期。
stacked_area = (
    alt.Chart(daily_counts)
    .mark_area(interpolate="linear", point=True)
    .encode(
        x=alt.X("date:T").title("Date (local)").axis(format="%m-%d", tickCount=4),
        y=alt.Y("sum(completed):Q")
        .stack("zero")
        .title("Completed requests")
        .scale(domain=[0, 55]),
        color=method_color,
        order=alt.Order("method:N").sort("ascending"),
        tooltip=[
            alt.Tooltip("date:T", title="Date", format="%Y-%m-%d"),
            alt.Tooltip("method:N", title="Method"),
            alt.Tooltip("sum(completed):Q", title="Component requests"),
        ],
    )
    .properties(width=420, height=220, title="Components within each daily total")
)

# 实际成图：A 在下、B 在上，09-04 的层厚为 16 和 24，上沿为 40。
# 上层基线随下层变化；在日期标记处悬停，读取对应的原始分量。
display(stacked_area)

alt.Chart(...)

## 7 矩形热力图：颜色也可以编码统计量

In [13]:
# 适用场景：矩形热力图用颜色比较“实现 × 负载”各组合的平均耗时。
# 数据组织：复用 requests；每格聚合两条请求，是分组均值表，不是相关矩阵。
# 最小绘制代码：mark_rect 用两个离散字段定位网格，color 编码 mean(latency_ms)。
# 常用参数调整：sort 固定负载和实现顺序；blues 是顺序色图，domain 固定 0～40 ms，
# 颜色此时表示数值，不再表示 A、B；tooltip 保留均值和样本量。
# 与另一张同指标热力图比较时，需核对颜色域；仅改颜色不改变聚合值。
latency_heatmap = (
    alt.Chart(requests)
    .mark_rect()
    .encode(
        x=alt.X("load:O")
        .sort(["Low", "Medium", "High"])
        .title("Load level")
        .axis(labelAngle=0),
        y=alt.Y("method:N").sort(["A", "B"]).title("Method"),
        color=alt.Color("mean(latency_ms):Q")
        .title("Mean latency (ms)")
        .scale(scheme="blues", domain=[0, 40]),
        tooltip=[
            "method:N",
            "load:O",
            alt.Tooltip("mean(latency_ms):Q", title="Mean latency (ms)", format=".1f"),
            alt.Tooltip("count():Q", title="Requests"),
        ],
    )
    .properties(width=360, height=140, title="Mean within each method and load")
)
print(requests.groupby(["method", "load"])["latency_ms"].agg(["size", "mean"]))

# 实际成图：每格样本量为 2；High 列 A=32.0 ms、B=24.5 ms，深浅不同。
# 颜色表示同一统计量，但均值隐藏了格内原始差异。
display(latency_heatmap)

               size  mean
method load              
A      High       2  32.0
       Low        2  10.0
       Medium     2  20.0
B      High       2  24.5
       Low        2   9.0
       Medium     2  16.0


alt.Chart(...)

## 8 直方图：用分箱观察耗时分布

In [14]:
# 适用场景：直方图观察耗时落入各区间的记录数，比较分箱宽度的影响。
# 数据组织：复用 requests；合并两种实现的 12 条记录，本例不再分组。
# 最小绘制代码：mark_bar、带 bin 的耗时轴与 count() 频数轴。
# 常用参数调整：bin(step=5) 指定 5 ms 箱宽；maxbins 是自动分箱的数量上限，
# 不是固定箱宽。已指定 step 时，maxbins 等自动选宽参数会被忽略。
histogram = (
    alt.Chart(requests)
    .mark_bar(color="#0072B2")
    .encode(
        x=alt.X("latency_ms:Q").bin(step=5).title("Latency (ms)"),
        y=alt.Y("count():Q").title("Requests").scale(domain=[0, 5]),
        tooltip=[alt.Tooltip("latency_ms:Q", bin=alt.Bin(step=5)), "count():Q"],
    )
    .properties(width=260, height=180, title="Bin width: 5 ms")
)
# 把 step 改为 10，生成较宽分箱供并排比较；悬停字段使用相同箱宽。
wide_histogram = histogram.encode(
    x=alt.X("latency_ms:Q").bin(step=10).title("Latency (ms)"),
    tooltip=[alt.Tooltip("latency_ms:Q", bin=alt.Bin(step=10)), "count():Q"],
).properties(title="Bin width: 10 ms")

# 实际成图：两图描述同样 12 条记录，箱宽改变细节，柱高仍是记录数。
display(alt.hconcat(histogram, wide_histogram))

alt.HConcatChart(...)

## 9 用图层和配置组织展示

alt.layer 将图叠在同一坐标中；这里给均值柱添加数字标签，标签仍采用相同的聚合表达式。alt.value 表示固定值，用它把标签统一设为深灰色；alt.hconcat 则保留独立面板。

configure_axis、configure_view 等配置作用于整个图表。组合图应在组合完成后设置顶层配置，避免把带配置的子图继续拿来拼接。

In [15]:
mean_labels = mean_bars.mark_text(align="left", dx=5, color="#333333").encode(
    text=alt.Text("mean(latency_ms):Q", format=".1f"),
    color=alt.value("#333333"),
)
labeled_means = (
    alt.layer(mean_bars, mean_labels)
    .encode(
        x=alt.X("mean(latency_ms):Q").title("Mean latency (ms)").scale(domain=[0, 25])
    )
    .configure_axis(labelFontSize=12, titleFontSize=12, gridColor="#E6E6E6")
    .configure_view(stroke=None)
)
display(labeled_means)
# 标签与柱子读取同一个均值；预留横轴空间，使末端数字不会紧贴绘图区边界。

alt.LayerChart(...)

## 10 变量参数与阈值过滤

参数（parameter）让图表在浏览器中保存可变化的状态。变量参数保存一个值；绑定控件后，读者可以改变它。binding_range 创建滑块，param 的 value 给出初始值，add_params 把参数加入图表。

alt.datum.latency_ms 表示图表当前记录的耗时字段；transform_filter 只保留条件为真的记录。这里先保持全部请求，再用滑块选择耗时上限。

In [16]:
latency_limit = alt.param(
    name="latency_limit",
    value=40,
    bind=alt.binding_range(min=5, max=40, step=1, name="Latency limit (ms): "),
)
threshold_chart = (
    scatter.add_params(latency_limit)
    .transform_filter(alt.datum.latency_ms <= latency_limit)
    .properties(title="Keep requests below the latency limit")
)
display(threshold_chart)
# 将滑块调到 20：只剩 A 的 3 个点、B 的 4 个点；调回 40 恢复全部 12 个点。
# 浏览器过滤不会改写 Python 中的 requests 表，也不需要重新运行单元。

alt.Chart(...)

## 11 点选择与条件高亮

选择参数保存通过点击或拖动得到的数据查询。selection_point 按离散值选择；fields=["method"] 表示点击一个点时选中其实现类别，而非仅选中这条请求。toggle=False 使每次点击只保留最新类别。

条件编码（condition）决定满足条件与不满足条件时如何显示。这里采用当前推荐的 when → then → otherwise 写法，alt.value 表示固定值；高亮只改变透明度，未选点仍然保留。

In [17]:
method_pick = alt.selection_point(
    name="method_pick", fields=["method"], toggle=False, clear="dblclick", empty=True
)
category_pick = (
    scatter.add_params(method_pick)
    .encode(opacity=alt.when(method_pick).then(alt.value(1)).otherwise(alt.value(0.15)))
    .properties(title="Click a point to highlight its method")
)
display(category_pick)
# 点击 A 的任一点后，A 全组突出、B 变淡；点击 B 会切换，双击绘图区清除选择。
# empty=True：未选择或清除选择时，所有记录都通过条件，因而全部显示为不透明。

alt.Chart(...)

## 12 区间选择与空选择

selection_interval 用拖动框选连续范围，也常称为 brush。把它接入条件颜色后，框内点保持类别颜色，框外点变灰。这里设置 empty=False，因此刚打开图时全部为灰色；双击清除后也回到这个状态。

空选择表示还没有范围。已经画出一个不包含任何点的范围，则是有效范围内恰好没有观测；应把两种情况区分开。

In [18]:
detail_brush = alt.selection_interval(
    name="detail_brush", encodings=["x", "y"], empty=False, clear="dblclick"
)
brush_chart = (
    scatter.add_params(detail_brush)
    .encode(
        color=alt.when(detail_brush).then(method_color).otherwise(alt.value("#BBBBBB"))
    )
    .properties(title="Drag a rectangle; empty selection highlights none")
)
display(brush_chart)
# 拖动矩形后只突出范围内的点；双击清除后全部恢复灰色。
# 颜色变灰并没有删除数据，坐标仍显示完整输入范围。

alt.Chart(...)

## 13 选择如何驱动另一张图

transform_filter 也可直接接收选择参数。下面在散点图上收集选择，在另一张图中过滤，再按实现计数。视图级的过滤先执行，编码中的 count() 随后处理剩余记录。

这里选择 empty=True，使初始状态显示全部。横轴计数固定为 0 到 6，类别域固定为 A、B，便于比较选择前后的变化。

In [19]:
linked_brush = alt.selection_interval(name="linked_brush", empty=True)
linked_points = scatter.add_params(linked_brush).properties(
    width=330, title="Drag to select"
)
linked_counts = (
    alt.Chart(requests)
    .transform_filter(linked_brush)
    .mark_bar()
    .encode(
        x=alt.X("count():Q").title("Selected requests").scale(domain=[0, 6]),
        y=alt.Y("method:N").title("Method").scale(domain=["A", "B"]),
        color=method_color,
        tooltip=["method:N", "count():Q"],
    )
    .properties(width=230, height=240, title="Count after filtering")
)
display(alt.hconcat(linked_points, linked_counts))
# 初始 A、B 各 6 条；只框住耗时大于 25 ms 的点时，计数分别为 2、1。
# 在没有点的位置框选时，两组都没有柱；双击散点图清除后恢复各 6 条。

alt.HConcatChart(...)

## 14 补充练习：散点图与计数条形图联动

以下是本节导出页面的两种实际状态：初始 A、B 各有 6 条记录，拖选约 0～35 kB 后各保留 3 条。运行后可自己拖选并双击清除。

![Altair 初始状态：全部请求及 A、B 各计数 6](image/04-altair-all.png)

![Altair 选择状态：输入范围约 0～35 kB，A、B 各计数 3](image/04-altair-selected.png)

In [20]:
# 补充练习：复用 requests、scatter 与 method_color，组合已学的选择和计数。
# encodings=["x"] 只筛选输入大小，纵向拖动不会额外筛选耗时。
# 左图保留所有请求并淡化未选点，右图先过滤再计数；无选择时显示全部。
report_brush = alt.selection_interval(
    name="report_brush", encodings=["x"], clear="dblclick", empty=True
)
report_points = (
    scatter.add_params(report_brush)
    .encode(
        opacity=alt.when(report_brush).then(alt.value(1)).otherwise(alt.value(0.16))
    )
    .properties(
        width=380, height=250, title="a  Drag across payload; double-click to reset"
    )
)
report_counts = (
    alt.Chart(requests)
    .transform_filter(report_brush)
    .mark_bar()
    .encode(
        x=alt.X("count():Q")
        .title("Selected requests")
        .scale(domain=[0, 7])
        .axis(tickMinStep=1),
        y=alt.Y("method:N").title("Method").scale(domain=["A", "B"]),
        color=method_color,
        tooltip=["method:N", "count():Q"],
    )
    .properties(width=230, height=250, title="b  Counts within the selected range")
)
# 右图先按 payload 的选择过滤，再计数；不把柱高解释为平均耗时。

# 常用调整：数字标签使用相同 count() 口径；组合完成后设置整体字号和网格。
report_labels = report_counts.mark_text(align="left", dx=5).encode(
    text=alt.Text("count():Q", format="d"), color=alt.value("#333333")
)
report = (
    alt.hconcat(report_points, alt.layer(report_counts, report_labels), spacing=30)
    .properties(title="Simulated requests: inspect a payload range")
    .configure_axis(
        labelFontSize=12, titleFontSize=12, gridColor="#E6E6E6", domainColor="#777777"
    )
    .configure_legend(labelFontSize=12, titleFontSize=12)
    .configure_title(fontSize=14, anchor="start")
    .configure_view(stroke=None)
    .configure(background="white", font="Arial")
)
display(report)
# 拖选输入大小约 0～35 kB：两组各 3 条；选择约 35～55 kB：两组各 2 条。
# 左侧变淡的点仍保留上下文；右侧的计数只使用选中记录。双击左图恢复各 6 条。

alt.HConcatChart(...)

### 14.1 对照所选范围的原始记录

In [21]:
# 复用 requests，用 0～35 kB 避开观测点边界，查看图中计数对应哪些记录。
selected_rows = requests.loc[requests["payload_kb"].between(0, 35)]
print(
    selected_rows[["request_id", "method", "payload_kb", "latency_ms"]].to_string(
        index=False
    )
)  # 预期：六行 r01/r02/r03/r07/r08/r09，两组各含 10/20/30 kB 的记录。
print("按实现计数：")
print(selected_rows.groupby("method").size())
# 两组各 3 条；与联动图选择相同输入范围后的柱高对应。

request_id method  payload_kb  latency_ms
       r01      A          10           8
       r02      A          20          12
       r03      A          30          17
       r07      B          10           7
       r08      B          20          11
       r09      B          30          14
按实现计数：
method
A    3
B    3
dtype: int64


## 15 保存 HTML 并检查资源依赖

save 保存 HTML 时，默认从在线 CDN 加载 Vega、Vega-Lite 与 vegaEmbed 的 JavaScript。文件较小，但首次加载需要网络。inline=True 将这些运行库嵌入文件，需要已经安装 vl-convert-python，文件也会更大。

内嵌运行库不等于自动下载所有远程数据或图片。本例传入本地 DataFrame，数据随规格保存，也没有引用远程图片，因此适合独立离线阅读。浏览器中的框选和过滤不需要持续运行 Python 服务。

下面把结果写入本次新建的临时目录。文件保留到手动清理；重复运行会创建新目录。

堆叠、面积和热力图也分别保存为内嵌资源 HTML，便于打开后逐一悬停核对。export_paths 记录本次实际生成的文件，供后面的清理单元使用。

In [22]:
import tempfile
from pathlib import Path

export_dir = Path(tempfile.mkdtemp(prefix="altair-ch04-"))
cdn_path = export_dir / "requests-cdn.html"
inline_path = export_dir / "requests-inline.html"
# 同一个 report 分别导出 CDN 版和内嵌版，比较资源依赖与文件大小。
report.save(cdn_path, embed_options={"renderer": "svg", "actions": False})
report.save(
    inline_path, inline=True, embed_options={"renderer": "svg", "actions": False}
)
export_paths = [cdn_path, inline_path]
# 其余图形按名称逐一导出，文件都保留在这次创建的临时目录。
chart_exports = {
    "stacked-bars": stacked_bars,
    "normalized-bars": normalized_bars,
    "total-area": total_area,
    "stacked-area": stacked_area,
    "latency-heatmap": latency_heatmap,
}
for name, chart in chart_exports.items():
    chart_path = export_dir / f"{name}.html"
    chart.save(
        chart_path, inline=True, embed_options={"renderer": "svg", "actions": False}
    )
    export_paths.append(chart_path)
# 汇总所有导出位置；打开 HTML 后再按本章方式清理。
for path in export_paths:
    print(path)
    print(f"大小：{path.stat().st_size:,} bytes")
# renderer='svg' 指 HTML 页面的交互绘制方式；此处导出的仍是可交互 HTML 文件。
# requests 的两个文件包含同一个图表规格，资源加载方式不同；另有五份图形示例。

C:\Users\ZHUANG\AppData\Local\Temp\altair-ch04-rwpk_3ts\requests-cdn.html
大小：5,549 bytes
C:\Users\ZHUANG\AppData\Local\Temp\altair-ch04-rwpk_3ts\requests-inline.html
大小：902,575 bytes
C:\Users\ZHUANG\AppData\Local\Temp\altair-ch04-rwpk_3ts\stacked-bars.html
大小：900,176 bytes
C:\Users\ZHUANG\AppData\Local\Temp\altair-ch04-rwpk_3ts\normalized-bars.html
大小：900,226 bytes
C:\Users\ZHUANG\AppData\Local\Temp\altair-ch04-rwpk_3ts\total-area.html
大小：899,949 bytes
C:\Users\ZHUANG\AppData\Local\Temp\altair-ch04-rwpk_3ts\stacked-area.html
大小：900,190 bytes
C:\Users\ZHUANG\AppData\Local\Temp\altair-ch04-rwpk_3ts\latency-heatmap.html
大小：900,828 bytes


打开 requests-inline.html，先看两组各为 6，再拖选 0～35 kB，观察计数变为各 3；悬停读取请求字段，双击恢复全部。

其他文件可分别练习读取分量、占比和颜色：09-01 完成 20 次、A 为 12，09-07 完成 50 次、A 为 30，两天 A 都占 60%。热力图 High 列的 A、B 均值为 32.0、24.5 ms，每格有 2 条记录。

内嵌版可在无缓存、断网后重新加载；CDN 版仍需取得在线脚本，已有缓存可能掩盖这一依赖。若输入改为远程 URL，数据也需联网加载。

查看或复制完成后，可把 cleanup_exports 改为 True，删除本次七份 HTML 和空目录。目录另有文件时会保留；需要继续使用时保持 False。

In [23]:
cleanup_exports = False
if cleanup_exports:
    for path in export_paths:
        path.unlink(missing_ok=True)
    try:
        export_dir.rmdir()
        print("已删除本次导出文件和空目录。")  # 预期：启用清理且目录成功删除时显示。
    except OSError:
        print("已删除记录中的导出文件；目录仍保留，请检查其他内容：", export_dir)  # 预期：目录删除失败时显示本次路径，提醒检查仍保留的内容。
else:
    print("保留本次导出：", export_dir)  # 预期：默认不清理，显示本次 altair-ch04- 临时目录。
# 只清理 export_paths 记录的已知文件；先复制需要分享的文件。

保留本次导出： C:\Users\ZHUANG\AppData\Local\Temp\altair-ch04-rwpk_3ts


## 本章小结

Altair 用数据、图元和编码描述图表；Q、N、O、T 决定字段如何参与尺度和图形表达。聚合前先确定分组，分箱前先确定希望保留的分布细节。

堆叠图中分量读厚度，总量读上沿；normalize 改成组内占比，同时隐藏总量差异。面积图要核对时间间距与插值含义；矩形热力图用颜色表达明确的统计量，不能混淆定量色图与类别配色。

参数把交互状态加入规格。条件编码改变外观，过滤改变后续参与统计的记录；选择方式、空选择行为和固定尺度共同决定读者怎样理解变化。

HTML 导出需要同时考虑 JavaScript 运行库与数据来源。内嵌资源、本地数据和实际离线复查一起构成本例的独立阅读条件。

## 练习

（1）不等距输入应该怎样编码

把两组输入大小都改为 10、20、30、60、120、240 kB，分别以 Q 和 N 绘制同一批耗时。说明哪张图保留输入大小的数值间隔，哪张图更像逐项比较；为分析输入大小与耗时的关系选择一种，并说明理由。

In [24]:
exercise_data = requests.copy()
exercise_data["payload_kb"] = [10, 20, 30, 60, 120, 240] * 2
exercise_chart = None
# 在此完成两种编码，使用 alt.hconcat 并排比较；重新设置合适的横轴范围。

if exercise_chart is not None:
    display(exercise_chart)  # 预期：完成后应显示数量型与有序型横轴的并排对照，核对间距差异。

（2）先选择，再计算均值

把综合图右侧的记录数改为选中请求的平均耗时，悬停中同时保留样本量。保留左侧完整上下文，并说明是否还适合沿用 0 到 7 的横轴范围。用 0～35 kB 的表格结果核对均值；讨论 empty=False 会怎样改变初始画面。

In [25]:
exercise_mean_chart = None
# 在此创建独立选择参数、散点图与均值图；右图先 transform_filter，再聚合。
# 请为平均耗时设置带 ms 的标题，并选取对应数值范围。

if exercise_mean_chart is not None:
    display(exercise_mean_chart)  # 预期：完成后应显示散点选择与右侧均值联动，右轴单位为 ms。
print(
    requests.loc[requests["payload_kb"].between(0, 35)]
    .groupby("method")["latency_ms"]
    .mean()
)

method
A    12.333333
B    10.666667
Name: latency_ms, dtype: float64


（3）为断网分享选择导出方式

选择一张自己完成的图，在新的临时目录保存内嵌资源版 HTML。检查图中的数据是否也在文件内，并在禁用缓存、断网条件下测试交互。说明若把输入改为在线 CSV，为什么仅设置 inline=True 还不够。

In [26]:
exercise_export = None
# 在此选择图表、创建新的临时目录，并调用 save(..., inline=True)。
# 完成浏览器检查后，在下面记录观察；不要把成功保存文件当成离线测试通过。
offline_observation = "待实际检查后填写"

print("导出路径：", exercise_export)  # 预期：未填写时为 None；完成练习后应显示自己生成的文件路径。
print("离线观察：", offline_observation)  # 预期：未填写时显示待检查提示；完成后应是实际观察记录。

导出路径： None
离线观察： 待实际检查后填写


（4）总量变大后，占比会怎样变化

将 09-07 的 A、B 完成量都乘以 2，分别重画普通堆叠柱与百分比堆叠柱，并为数量图重新选择纵轴范围。核对总量从 50 变为 100，而 A 仍占 60%。说明两种图各自突出或隐藏的信息，并解释为什么不能把原始汇总表改用 count()。

In [27]:
exercise_daily = daily_counts.copy()
last_date = exercise_daily["date"] == pd.Timestamp("2026-09-07")
exercise_daily.loc[last_date, "completed"] *= 2
exercise_composition = None
# 在此使用 exercise_daily 创建两张图；数量轴应能显示总数 100。
# 核对：其他记录日不变；百分比图的四根柱与原例组成相同。

if exercise_composition is not None:
    display(exercise_composition)  # 预期：完成后核对数量图与比例图的差异，并与题目要求比较。

## 练习提示与解析

以下对应练习（1）。先独立作答，卡住时依次查看提示，完成后再对照解析。

提示 1：字段保存为数字，不意味着它在所有问题中都应使用同一种编码。

提示 2：比较相邻横坐标差，判断图上的间距是否仍与输入量差成比例。

### 练习（1）参考解析

若 Q 使用线性尺度，相邻输入差为 10、10、30、60、120 kB，对应间距比为 1∶1∶3∶6∶12；相同输入下的两组耗时仍应对齐。N 将这些数当成六个类别，类别间通常等距，适合逐项对比，却丢失输入量的数值间隔。

分析输入大小与耗时关系时选 Q，并设置能覆盖 10～240 kB 的横轴范围。若只比较六个指定配置，也可用 N，但应明确它表达类别比较。编码类型应由问题与字段含义决定，不能只依据底层数据类型。

## 参考与引用来源

下列官方页面在写作前核查，示例按 Altair 6.3.0 执行。数据、示例和练习为本章自制。补充时官网部分示例页仍标为 6.2.2，相关接口均在本章 6.3.0 环境重新执行。

| 网站 | 本章参考内容与定位 |
| --- | --- |
| Vega-Altair 官方文档 | [Overview](https://altair-viz.github.io/getting_started/overview.html)：声明式映射；[Specifying Data](https://altair-viz.github.io/user_guide/data.html)：表格记录与 DataFrame 输入；[Marks](https://altair-viz.github.io/user_guide/marks/index.html)：点、柱、线、文本图元；[Bar](https://altair-viz.github.io/user_guide/marks/bar.html#stacked-bar-chart)：颜色分组与堆叠柱；[Normalized Stacked Bar Chart](https://altair-viz.github.io/gallery/normalized_stacked_bar_chart.html)：stack=normalize；[Stack](https://altair-viz.github.io/user_guide/transform/stack.html)：堆叠起止值与 zero、normalize；[Channels: Order](https://altair-viz.github.io/user_guide/encodings/channels.html#order)：堆叠次序；[Area](https://altair-viz.github.io/user_guide/marks/area.html) 的 Area Chart、Overlaying Lines and Point Markers、Stacked Area Chart：时间变化、点标记与分量；[Simple Stacked Area Chart](https://altair-viz.github.io/gallery/simple_stacked_area_chart.html)：声明式面积示例；[Rect: Heatmap](https://altair-viz.github.io/user_guide/marks/rect.html#heatmap)：矩形图元与离散行列；[Axis](https://altair-viz.github.io/user_guide/generated/core/altair.Axis.html) 的 format：数值百分比与日期显示；[Line](https://altair-viz.github.io/user_guide/marks/line.html#line-chart-with-point-markers)：分组折线与 point=True；[Encodings](https://altair-viz.github.io/user_guide/encodings/index.html#encoding-data-types)：Q、N、O、T 与类型影响；[Channel Options](https://altair-viz.github.io/user_guide/encodings/channel_options.html#x-and-y)：字段、排序、尺度与标题；[Times & Dates](https://altair-viz.github.io/user_guide/times_and_dates.html#altair-and-pandas-datetimes)：日期及本地时间；[Aggregate](https://altair-viz.github.io/user_guide/transform/aggregate.html)：编码聚合、分组与显式聚合；[Bin](https://altair-viz.github.io/user_guide/transform/bin.html) 和 [BinParams](https://altair-viz.github.io/user_guide/generated/core/altair.BinParams.html)：分箱、step、maxbins；[Layered & Multi-View Charts](https://altair-viz.github.io/user_guide/compound_charts.html)：图层与横向组合；[Customizing Visualizations](https://altair-viz.github.io/user_guide/customization.html)：顶层配置、颜色域、尺寸；[Parameters, Conditions, & Filters](https://altair-viz.github.io/user_guide/interactions/parameters.html)：参数、when 条件与联动；[Bindings & Widgets](https://altair-viz.github.io/user_guide/interactions/bindings_widgets.html#data-driven-comparisons)：滑块与 datum 比较；[selection_point](https://altair-viz.github.io/user_guide/generated/api/altair.selection_point.html) 与 [selection_interval](https://altair-viz.github.io/user_guide/generated/api/altair.selection_interval.html)：字段选择、toggle、clear、empty；[Filter](https://altair-viz.github.io/user_guide/transform/filter.html)：表达式与选择过滤；[Displaying Altair Charts](https://altair-viz.github.io/user_guide/display_frontends.html#altair-s-renderer-framework)：默认 HTML 展示与在线资源；[Saving Altair Charts](https://altair-viz.github.io/user_guide/saving_charts.html)：JSON、HTML、Offline HTML support 与 Additional Dependencies。 |
| Vega-Lite 官方文档 | [Transformation](https://vega.github.io/vega-lite/docs/transform.html)：视图变换先于编码内聚合，视图变换依声明顺序执行；[Stack](https://vega.github.io/vega-lite/docs/stack.html) 的 Stack in Encoding Field Definition、Normalized Stacked Bar and Area Charts、Sorting Stack Order：堆叠方式、归一化与次序。 |
| Python 3.12 官方文档 | [tempfile.mkdtemp](https://docs.python.org/3.12/library/tempfile.html#tempfile.mkdtemp)：临时目录与手动清理；[Path.stat](https://docs.python.org/3.12/library/pathlib.html#pathlib.Path.stat)：读取文件大小；[Path.unlink](https://docs.python.org/3.12/library/pathlib.html#pathlib.Path.unlink) 与 [Path.rmdir](https://docs.python.org/3.12/library/pathlib.html#pathlib.Path.rmdir)：删除已知文件与空目录。 |
| pandas 3.0.6 官方文档 | [to_datetime](https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html)：本地日期列；[Series.dt.strftime](https://pandas.pydata.org/docs/reference/api/pandas.Series.dt.strftime.html)：日期标签的格式化；[Group by: Aggregation](https://pandas.pydata.org/docs/user_guide/groupby.html#aggregation)：按日求和与分组均值；[Timestamp](https://pandas.pydata.org/docs/reference/api/pandas.Timestamp.html)：练习中的日期标量；表格数据与模拟规则为本章自定。 |